In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import PromptTemplate

from langchain.chains import RetrievalQA

from langchain.embeddings import (HuggingFaceEmbeddings)


In [3]:
## Read the ppdfs from the folder
loader=PyPDFDirectoryLoader("./us_census")

documents=loader.load()

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

final_documents=text_splitter.split_documents(documents)
final_documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census\\acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.1 Public policy changes included \nthe renewal of the Public Health Emergency, wh

In [4]:
len(final_documents)

316

In [5]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")

# # Encode text into embeddings
# text = "This is a sample sentence."
# embedding = model.encode(text, normalize_embeddings=True)

# print(embedding)


In [6]:
Olama_embeddings = OllamaEmbeddings(model="paraphrase-multilingual")

In [7]:
## Embedding Using Huggingface
huggingface_embeddings=HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",      #sentence-transformers/all-MiniLM-l6-v2
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True}
)

C:\Users\kosha\AppData\Local\Temp\ipykernel_12652\3061680995.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings=HuggingFaceBgeEmbeddings(
c:\Users\kosha\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import  numpy as np
print(np.array(Olama_embeddings.embed_query(final_documents[0].page_content)))
print(np.array(Olama_embeddings.embed_query(final_documents[0].page_content)).shape)

[-4.80433140e-02  6.39138500e-02 -2.00987510e-03 -7.99880550e-02
  5.20382730e-03  1.80788970e-02  2.26321870e-02 -2.61098740e-02
  3.88821800e-02  3.05576020e-03  6.40970600e-02 -2.26245200e-02
  7.84825300e-03  5.63629340e-02 -3.46360470e-02 -5.07643700e-02
  1.57488980e-02  3.63850500e-02 -6.02876840e-02  3.82128800e-02
 -4.35190420e-02 -3.57583100e-02 -2.52000690e-02  2.94612030e-02
  7.17714480e-03  3.70091160e-02 -3.74764760e-04 -2.82812730e-02
  1.63843690e-02  1.06889100e-02  7.12723360e-02  4.28362820e-02
  4.10492760e-02 -9.23350200e-02  3.99653470e-03  6.25559830e-03
  6.82255770e-03  5.63784700e-02  7.65815470e-03 -2.34565920e-02
  1.35975840e-02 -7.70803000e-02 -3.15404940e-02 -8.90023800e-03
 -1.91018700e-02  2.32452900e-02 -3.52726880e-02 -2.19946350e-02
 -6.58261760e-04 -2.82424520e-02  9.97852200e-03  1.74727920e-02
 -4.82273440e-02 -2.44446770e-02  1.68445480e-02 -3.52654700e-03
  8.50006700e-03 -4.36626750e-02  9.40319100e-03 -1.57038870e-02
  2.33140030e-03  2.96361

In [9]:
## VectorStore Creation
vectorstore=FAISS.from_documents(final_documents,Olama_embeddings)

In [10]:
## Query using Similarity Search
query="WHAT IS HEALTH INSURANCE COVERAGE?"
relevant_docments=vectorstore.similarity_search(query)

print(relevant_docments[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [11]:
retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":3})
print(retriever)

tags=['FAISS', 'OllamaEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025C18AE8160> search_kwargs={'k': 3}


In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['HUGGINGFACEHUB_API_TOKEN']= os.getenv("HUGGINGFACEHUB_API_TOKEN_1")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACEHUB_API_TOKEN_1")

The Hugging Face Hub is an platform with over 350k models, 75k datasets, and 150k demo apps (Spaces), all open source and publicly available, in an online platform where people can easily collaborate and build ML together.

In [23]:
from langchain_community.llms import HuggingFaceHub

hf=HuggingFaceHub(
    repo_id= "gpt2", #"mistralai/Mistral-7B-v0.1",  # Corrected repo_id
    task="text-generation",
    model_kwargs={"temperature":0.3,"max_length":500},
    huggingfacehub_api_token = HUGGINGFACE_API_KEY  # Ensure API key is used
)
query="What is the health insurance coverage?"
hf.invoke(query)

c:\Users\kosha\anaconda3\envs\llm\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


"What is the health insurance coverage?\n\nThe health insurance coverage is a form of insurance that covers a person's medical expenses.\n\nWhat is the cost of health care?\n\nThe cost of health care is the cost of the health care that is covered by the health insurance.\n\nWhat is the cost of insurance?\n\nThe cost of insurance is the cost of the health care that is covered by the insurance.\n\nWhat is the cost of insurance?\n\nThe cost of insurance is the cost of the health care that is covered by the insurance.\n\nWhat is the cost of insurance?\n\nThe cost of insurance is the cost of the health care that is covered by the insurance.\n\nWhat is the cost of insurance?\n\nThe cost of insurance is the cost of the health care that is covered by the insurance.\n\nWhat is the cost of insurance?\n\nThe cost of insurance is the cost of the health care that is covered by the insurance.\n\nWhat is the cost of insurance?\n\nThe cost of insurance is the cost of the health care that is covered by

In [24]:
# from langchain_huggingface import HuggingFacePipeline
# hf = HuggingFacePipeline.from_model_id(
#     model_id="mistralai/Mistral-7B-v0.1",
#     task="text-generation",
#     pipeline_kwargs={"temperature": 0, "max_new_tokens": 300},
# )

In [25]:
# #Hugging Face models can be run locally through the HuggingFacePipeline class.
# from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

# hf = HuggingFacePipeline.from_model_id(
#     model_id="mistralai/Mistral-7B-v0.1",
#     task="text-generation",
#     pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
# )

# llm = hf 
# llm.invoke(query)

In [26]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """

In [27]:
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])

In [28]:
retrievalQA=RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt":prompt}
)

In [29]:
query="""DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022"""

In [30]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})
print(result['result'])

c:\Users\kosha\anaconda3\envs\llm\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)



Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

erage (78.4 percent) in 2022, 
while New Mexico had the low-
est private coverage rate (54.4 
percent) (Figure 3).9
• Utah had the lowest rate of 
public coverage in 2022 (22.2 
percent), and New Mexico had 
the highest (Figure 4). 
• Twenty-seven states had lower 
uninsured rates in 2022 com-
pared with 2021. Maine was the 
only state whose uninsured rate 
increased (6.6 percent in 2022, 
up from 5.7 percent in 2021) 
(Figure 1 and Appendix Table 
B-1).
• From 2021 to 2022, 13 states 
reported increases in public cov-
erage, with only Rhode Island 
reporting a decrease of 2.2 
percentage points (Appendix 
Table B-3).
8 The Current Population Survey Annual 
Social and Economic Supplement (CPS 
ASEC) is the leading source of national level 
estimates of health insurance coverage. For a 
comparison of ACS and CPS ASEC measures 
of health insurance coverage, refer 